In [98]:
import os

# Path to the directory
folder_path = '../../../CASAS dataset/data/'

# Get a list of all CSV files in the directory
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# Extract file names without the '.csv' extension
file_names_without_extension = [os.path.splitext(f)[0] for f in csv_files]
csv_files

In [99]:
import pandas as pd
import datetime
import numpy as np

# Read CSV file
column_names = ['date', 'timestamp', 'location_name', 'detection']

entropy_df_list = []
COUNT = 0
for file_name in file_names_without_extension:
    COUNT +=1
    print(COUNT)
    id_pick = file_name
    print(id_pick)
    df = pd.read_csv('../../../CASAS dataset/data/'+file_name+'.csv', names=column_names)
    df = df.replace({'\t': ''}, regex=True)

    # only keep the location sensors- ON (ignore windows)
    df_on = df.query('detection == "ON"').copy()
    df_on['timestamp'] = pd.to_datetime(df_on['date'] + ' ' + df_on['timestamp'])
    df_on = df_on.drop(columns=['detection']).reset_index(drop=True).copy()
    
    
    ######## calculate for each person and each date
    
    date_array = df_on.date.unique()
    act_df_person = df_on
    entropy_rate_list = []
    date_list = [] 
    
    
    for target_date in date_array:
        date_list.append(target_date)
        
        # in each day
        filtered_df = act_df_person[act_df_person['date'] == target_date]        
    
        filtered_df = filtered_df.sort_values('timestamp').reset_index(drop=True)
        
        filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
        
        # Sort by timestamp
        filtered_df = filtered_df.sort_values('timestamp').reset_index(drop=True)
        
    ##################### calculate bout duration ##########################
        # Identify chunks of consecutive same state
        filtered_df['location_change'] = (filtered_df['location_name'] != filtered_df['location_name'].shift()).cumsum()
        
        # Group by each chunk
        chunk_durations = filtered_df.groupby(['location_change', 'location_name']).agg(
            start_time=('timestamp', 'first'),
            end_time=('timestamp', 'last')
        ).reset_index()
        chunk_durations = chunk_durations.dropna(subset=['start_time']).reset_index()
        
        # Calculate duration in minutes per chunk
        chunk_durations['duration_min'] = (chunk_durations['end_time'] - chunk_durations['start_time']).dt.total_seconds() / 60
        
        # Your sequence of locations
        locations = chunk_durations.location_name
        
        # Create a list of unique locations (states)
        unique_locations = sorted(set(locations))
        # if there is no transition
        if len(unique_locations) <= 1:  
            entropy_rate = np.NAN
        else:    
            # Initialize the transition matrix with zeros
            transition_matrix = pd.DataFrame(0, index=unique_locations, columns=unique_locations)
            
            # Count the transitions
            for i in range(len(locations) - 1):
                from_location = locations[i]
                to_location = locations[i + 1]
                transition_matrix.at[from_location, to_location] += 1
            
            # Convert counts to probabilities by dividing each row by its sum
            transition_matrix = transition_matrix.div(transition_matrix.sum(axis=1), axis=0)
            
            # if the transition matrix contain NAN
            if transition_matrix.isna().values.any():
                entropy_rate = np.NAN
                
            else:      
                # Step 4: Estimate stationary distribution π
                # Method: left eigenvector of P.T corresponding to eigenvalue 1
                # I have checked the result with another function (https://stephens999.github.io/fiveMinuteStats/stationary_distribution.html) the results are the same
                eigvals, eigvecs = np.linalg.eig(transition_matrix.T)
                stat_dist = np.real(eigvecs[:, np.isclose(eigvals, 1)])
                stat_dist = stat_dist[:, 0]
                stat_dist = stat_dist / stat_dist.sum()
                stat_dist = pd.DataFrame({'stat_dist': stat_dist, 'unique_locations': unique_locations}).set_index('unique_locations')       
        
                # Step 5: Compute entropy rate
                P = transition_matrix.copy()
                entropy_rate = 0.0
                for i in unique_locations:
                    for j in unique_locations:
                        if P.at[i, j] > 0:
                            entropy_rate -= stat_dist.at[i,'stat_dist'] * P.at[i, j] * np.log2(P.at[i, j])
        # normalize the original entropy rate by log2( N), N is the number of unique location 
        entropy_rate_normalized = entropy_rate / np.log2(len(unique_locations))
        entropy_rate_list.append(entropy_rate_normalized) 
            
    entropy_person = {
    'patient_id': id_pick,
    'date': date_list,
    'entropy_rate': entropy_rate_list 
    }
    
    entropy_person_df = pd.DataFrame(entropy_person)  
    entropy_df_list.append(entropy_person_df)

In [100]:
final_df = pd.concat(entropy_df_list, ignore_index=True)

final_df.to_csv('casas_output/activity_entropy_rates_ALL.csv', index=False)
final_df

# starting with "hh" were missing above

In [17]:
import pandas as pd
df_HC = pd.read_csv('casas_output/activity_entropy_rates_ALL.csv')

import os

# Path to the directory
folder_path = '../../../CASAS dataset/data/'

# Get a list of all CSV files in the directory
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# Extract file names without the '.csv' extension
file_names_without_extension = [os.path.splitext(f)[0] for f in csv_files]

missing_set = list(set(file_names_without_extension) - set(list(df_HC.patient_id.unique())))

In [2]:
missing_set

In [14]:
import pandas as pd
import datetime
import numpy as np

# Read CSV file
column_names = ['date', 'timestamp', 'location_name', 'detection']

entropy_df_list2 = []
COUNT = 0
for file_name in missing_set:
    COUNT +=1
    print(COUNT)
    id_pick = file_name
    print(id_pick)
    df = pd.read_csv('../../../CASAS dataset/data/'+file_name+'.csv', names=column_names, delimiter=' ')
    df = df.replace({'\t': ''}, regex=True)

    # only keep the location sensors- ON (ignore windows)
    df_on = df.query('detection == "ON"').copy()
    df_on['timestamp'] = pd.to_datetime(df_on['date'] + ' ' + df_on['timestamp'])
    df_on = df_on.drop(columns=['detection']).reset_index(drop=True).copy()
    
    
    ######## calculate for each person and each date
    
    date_array = df_on.date.unique()
    act_df_person = df_on
    entropy_rate_list = []
    date_list = [] 
    
    
    for target_date in date_array:
        date_list.append(target_date)
        
        # in each day
        filtered_df = act_df_person[act_df_person['date'] == target_date]        
    
        filtered_df = filtered_df.sort_values('timestamp').reset_index(drop=True)
        
        filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
        
        # Sort by timestamp
        filtered_df = filtered_df.sort_values('timestamp').reset_index(drop=True)
        
    ##################### calculate bout duration ##########################
        # Identify chunks of consecutive same state
        filtered_df['location_change'] = (filtered_df['location_name'] != filtered_df['location_name'].shift()).cumsum()
        
        # Group by each chunk
        chunk_durations = filtered_df.groupby(['location_change', 'location_name']).agg(
            start_time=('timestamp', 'first'),
            end_time=('timestamp', 'last')
        ).reset_index()
        chunk_durations = chunk_durations.dropna(subset=['start_time']).reset_index()
        
        # Calculate duration in minutes per chunk
        chunk_durations['duration_min'] = (chunk_durations['end_time'] - chunk_durations['start_time']).dt.total_seconds() / 60
        
        # Your sequence of locations
        locations = chunk_durations.location_name
        
        # Create a list of unique locations (states)
        unique_locations = sorted(set(locations))
        # if there is no transition
        if len(unique_locations) <= 1:  
            entropy_rate = np.NAN
        else:    
            # Initialize the transition matrix with zeros
            transition_matrix = pd.DataFrame(0, index=unique_locations, columns=unique_locations)
            
            # Count the transitions
            for i in range(len(locations) - 1):
                from_location = locations[i]
                to_location = locations[i + 1]
                transition_matrix.at[from_location, to_location] += 1
            
            # Convert counts to probabilities by dividing each row by its sum
            transition_matrix = transition_matrix.div(transition_matrix.sum(axis=1), axis=0)
            
            # if the transition matrix contain NAN
            if transition_matrix.isna().values.any():
                entropy_rate = np.NAN
                
            else:      
                # Step 4: Estimate stationary distribution π
                # Method: left eigenvector of P.T corresponding to eigenvalue 1
                # I have checked the result with another function (https://stephens999.github.io/fiveMinuteStats/stationary_distribution.html) the results are the same
                eigvals, eigvecs = np.linalg.eig(transition_matrix.T)
                stat_dist = np.real(eigvecs[:, np.isclose(eigvals, 1)])
                stat_dist = stat_dist[:, 0]
                stat_dist = stat_dist / stat_dist.sum()
                stat_dist = pd.DataFrame({'stat_dist': stat_dist, 'unique_locations': unique_locations}).set_index('unique_locations')       
        
                # Step 5: Compute entropy rate
                P = transition_matrix.copy()
                entropy_rate = 0.0
                for i in unique_locations:
                    for j in unique_locations:
                        if P.at[i, j] > 0:
                            entropy_rate -= stat_dist.at[i,'stat_dist'] * P.at[i, j] * np.log2(P.at[i, j])
        # normalize the original entropy rate by log2( N), N is the number of unique location 
        entropy_rate_normalized = entropy_rate / np.log2(len(unique_locations))
        entropy_rate_list.append(entropy_rate_normalized) 
            
    entropy_person = {
    'patient_id': id_pick,
    'date': date_list,
    'entropy_rate': entropy_rate_list 
    }
    
    entropy_person_df = pd.DataFrame(entropy_person)  
    entropy_df_list2.append(entropy_person_df)

In [24]:
add_df = pd.concat(entropy_df_list2, ignore_index=True)

df_combined_189 = pd.concat([df_HC, add_df], ignore_index=True)
df_combined_189.to_csv('casas_output/activity_entropy_rates_ALL_189.csv', index=False)
df_combined_189